# Import dan Load data

In [ ]:
# Instal Library
!pip install Sastrawi openpyxl

# Instal Library
!pip install Sastrawi openpyxl scikit-learn nltk


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: C:\Users\ruziq\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# Import Library
import pandas as pd
import re

import math
from collections import Counter

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_excel('TKI Sustainability Ecosystem.xlsx', sheet_name='IR')
df.head()

,Data
0,Konsep sustainability adalah suatu konsep pemb...
1,harusnya hutan disisakan buat ekosistem kehidu...
2,"badak Sumatra, harimau sumatera, gajah sumater..."
3,Kita harus menjaga ekosistem laut demi menjaga...
4,keren bang konten edukasi nya🔥 tentang karbon ...


# Preprocessing

In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)
documents = df['Data']

In [ ]:
# Case Folding
def case_folding(text):
    return text.lower()

In [ ]:
# Cleansing
def cleansing(text):
    text = re.sub(r'http\S+', '', text)         # hapus URL
    text = re.sub(r'@\w+', '', text)            # hapus mention
    text = re.sub(r'#\w+', '', text)            # hapus hashtag
    text = re.sub(r'\d+', '', text)             # hapus angka
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # hapus simbol & emoji
    text = re.sub(r'\s+', ' ', text).strip()    # hapus spasi berlebih

    return text

In [ ]:
# Tokenizing
def tokenizing(text):
    return text.split()

In [ ]:
# Normalization
normalization_dict = {

    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "utk": "untuk",
    "yg": "yang",
    "krn": "karena",
    "dr": "dari",
    "dgn": "dengan",
    "bgt": "banget",
    "tp": "tapi"

}

def normalization(tokens):

    normalized = []

    for token in tokens:

        if token in normalization_dict:
            normalized.append(normalization_dict[token])

        else:
            normalized.append(token)

    return normalized

In [ ]:
# Stopword Removal
factory_stopword = StopWordRemoverFactory()
stopwords = factory_stopword.get_stop_words()

def stopword_removal(tokens):

    filtered = []

    for token in tokens:

        if token not in stopwords:
            filtered.append(token)

    return filtered

In [ ]:
# Stemming
factory_stemmer = StemmerFactory()
stemmer = factory_stemmer.create_stemmer()

def stemming(tokens):

    hasil = []

    for token in tokens:
        hasil.append(stemmer.stem(token))

    return hasil

In [ ]:
# Preprocessing Pipeline
def preprocess(text):

    # ubah ke string
    text = str(text)

    # 1. case folding
    text = case_folding(text)

    # 2. cleansing
    text = cleansing(text)

    # 3. tokenizing
    tokens = tokenizing(text)

    # 4. normalization
    tokens = normalization(tokens)

    # 5. stopword removal
    tokens = stopword_removal(tokens)

    # 6. stemming
    tokens = stemming(tokens)

    return tokens

In [ ]:
# Proses Preprocessing
df['Preprocessing'] = documents.apply(preprocess)

In [ ]:
df[['Data', 'Preprocessing']].head()

,Data,Preprocessing
0,Konsep sustainability adalah suatu konsep pemb...,"[konsep, sustainability, suatu, konsep, bangun..."
1,harusnya hutan disisakan buat ekosistem kehidu...,"[harus, hutan, sisa, buat, ekosistem, hidup, s..."
2,"badak Sumatra, harimau sumatera, gajah sumater...","[badak, sumatra, harimau, sumatera, gajah, sum..."
3,Kita harus menjaga ekosistem laut demi menjaga...,"[jaga, ekosistem, laut, jaga, lestari, mahkluk..."
4,keren bang konten edukasi nya🔥 tentang karbon ...,"[keren, bang, konten, edukasi, nya, karbon, ma..."


In [ ]:
# Menyimpan hasil
df.to_excel('hasil_preprocessing.xlsx', index=False)

print("File hasil_preprocessing.xlsx berhasil di simpan!")

File hasil_preprocessing.xlsx berhasil di simpan!


# TF-IDF

In [ ]:
# Gabung token menjadi kalimat
df['Preprocessing_Text'] = df['Preprocessing'].apply(lambda x: ' '.join(x))
df[['Preprocessing_Text']].head()

,Preprocessing_Text
0,konsep sustainability suatu konsep bangun diri...
1,harus hutan sisa buat ekosistem hidup satwa ja...
2,badak sumatra harimau sumatera gajah sumatera ...
3,jaga ekosistem laut jaga lestari mahkluk hidup...
4,keren bang konten edukasi nya karbon mangrove ...


In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Preprocessing_Text'])

feature_names = vectorizer.get_feature_names_out()

sample_features = [
    'ekosistem',
    'lanjut',
    'industri',
]

# Create a dictionary to hold the columns for the new tfidf_df
data_for_tfidf_df = {}

# Iterate through each desired feature from sample_features
for feature in sample_features:
    if feature in feature_names:
        # If the feature exists in the vectorizer's vocabulary, get its TF-IDF scores
        feature_index = list(feature_names).index(feature)
        data_for_tfidf_df[feature] = tfidf_matrix[:, feature_index].toarray().flatten()
    else:
        # If the feature does not exist, create a column of zeros
        data_for_tfidf_df[feature] = [0.0] * tfidf_matrix.shape[0] # tfidf_matrix.shape[0] is the number of documents

# Create the DataFrame
tfidf_df = pd.DataFrame(data_for_tfidf_df).round(3)

print("MATRIKS HASIL TF-IDF:")
tfidf_df.head()

MATRIKS HASIL TF-IDF:


,ekosistem,lanjut,industri
0,0.000,0.000,0.000
1,0.126,0.000,0.000
2,0.052,0.000,0.000
3,0.067,0.000,0.000
4,0.062,0.000,0.000


In [ ]:
tfidf_df.to_excel("hasil_tfidf.xlsx", index=False)

print("File hasil_tfidf.xlsx berhasil disimpan")

File hasil_tfidf.xlsx berhasil disimpan


# Vector Space Model (VSM)

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('indonesian'))

In [ ]:
query = "ekosistem berkelanjutan industri"

query = query.lower()

query = re.sub(r'[^a-zA-Z\s]', '', query)

query = word_tokenize(query)

query = [word for word in query if word not in stop_words]

query = [stemmer.stem(word) for word in query]

query_processed = " ".join(query)

print(query_processed)

ekosistem lanjut industri


In [ ]:
# Hitung Cosine Similarity dan Ranking
query_vector = vectorizer.transform([query_processed])
similarity = cosine_similarity(
    query_vector,
    tfidf_matrix
)

similarity_df = pd.DataFrame({
    'Dokumen': [f'Doc {i+1}' for i in range(len(df))],
    'Similarity': similarity.flatten(),
    'Isi_Dokumen': df['Data']
})

ranking = similarity_df.sort_values(
    by='Similarity',
    ascending=False
)

In [ ]:
print("\nTop 10 Dokumen Paling Relevan")
display(ranking.head(10))


Top 10 Dokumen Paling Relevan


,Dokumen,Similarity,Isi_Dokumen
24,Doc 25,0.281,Peran warga dalam pengelolaan ekosistem mangro...
19,Doc 20,0.276,Status keberlanjutan ekosistem mangrove di Pul...
22,Doc 23,0.235,"Oleh karena itu, hal utama yang harus dikelola..."
37,Doc 38,0.232,Pengelolaan ekosistem wisata mangrove secara b...
40,Doc 41,0.182,Keberlanjutan ekosistem tanah juga bergantung ...
34,Doc 35,0.176,Model pengelolaan partisipasi warga memperkuat...
33,Doc 34,0.173,Keseimbangan peningkatan kesejahteraan ekonomi...
43,Doc 44,0.168,Manfaat konservasi tanah dan air pada ekosiste...
23,Doc 24,0.166,Pembangunan ekosistem laut yang berkelanjutan ...
11,Doc 12,0.164,Langkah kecil ini menjadi bagian dari upaya me...


In [ ]:
ranking = ranking.reset_index(drop=True)

ranking.index = ranking.index + 1
ranking.index.name = 'Ranking'

ranking['Similarity'] = ranking['Similarity'].round(3)

ranking.to_excel(
    'hasil_vsm.xlsx',
    index=True
)

print("File hasil_vsm.xlsx berhasil disimpan!")

File hasil_vsm.xlsx berhasil disimpan!
